In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Tries

A [trie](https://en.wikipedia.org/wiki/Trie) is a data structure that implements the abstract data type of a *dictionary* in the case that the keys are strings.  
The set $\mathbb{T}$ of all tries is defined inductively using the constructor  
$$
\mathtt{Trie}: \mathtt{Value} \times \mathtt{Array}(\Sigma) \times \mathtt{Array}(\mathbb{T}) \rightarrow \mathbb{T}.
$$
Here $\Sigma$ is the *alphabet*, i.e. the set of characters. The inductive definition of the set $\mathbb{T}$ has only a single clause: If  
- $v \in \mathtt{Value} \cup \{\Omega\}$,  
- $C\!s = [c_1, \cdots, c_n] \in \mathtt{Array}(\Sigma)$ is an array of different characters of length $n$, and  
- $T\!s = [t_1, \cdots, t_n] \in \mathtt{Array}(\mathbb{T})$ is an array of tries of the same length $n$,  

then we have  
$$
\mathtt{Trie}(v, C\!s, T\!s) \in \mathbb{T}.
$$


The class `Trie` defined below implements *tries*.  An object $t$ of the class `Trie` represents the *trie*
$$ \texttt{Trie}(t.\texttt{mValue}, t.\texttt{mChars}, t.\texttt{mTries}). $$
The constructor creates an empty *trie*.

In [ ]:
class Trie<T> {
  mValue: T | null;
  mChars: string[];
  mTries: Trie<T>[];
  mParent: string; 
  static sTrieCount: number = 0;

  constructor() {
    this.mValue = null;
    this.mChars = [];
    this.mTries = [];
    this.mParent = '';
  }
}     
interface Trie<T> {
  mID: string;
  mValue: T | null;
  mChars: string[];
  mTries: Trie<T>[];
  mParent: string;
  find(s: string): T | null;
  insert(s: string, v: T): void;
  isEmpty(): boolean;
  delete(s: string): void;
  allKeys(prefix?: string): Set<string>;
  replacePrefix(s: string, p: string): Set<string>;
  findPrefix(s: string): Set<string>;
  toDot(): Promise<string>;
  assignIDs(TrieDict: Record<string, Trie<T>>): void;
}

In order to specify the meaning of the class `Trie` we define the function `find`,
which can be used to retrieve the values stored in a trie.
This function has the signature
$$ \texttt{find}: \mathbb{T} \times \Sigma^* \rightarrow \texttt{Value} \cup \{ \Omega\}. $$
For a trie $t$ and a string $s$, the expression $t.\texttt{find}(s)$ returns the
value that is associated with the key $s$ in the trie $t$.  The expression
$\texttt{Trie}(v,C\!s,T\!s).\texttt{find}(s)$ is defined by induction on the length of the  string $s$:
  - $\texttt{Trie}(v, C\!s, T\!s).\texttt{find}(\varepsilon) = v$.
  
    The value associated with the empty string $\varepsilon$ is stored at the root of the trie.
  - $c = c_i \rightarrow 
     \texttt{Trie}(v, [c_1, \cdots, c_n], [t_1, \cdots, t_n]).\texttt{find}(cr) = t_i.\texttt{find}(r)
    $
    
    The trie $\texttt{Trie}(v, [c_1, \cdots, c_n], [t_1, \cdots, t_n])$ associates a value with
    the key $cr$ if the array $[c_1, \cdots, c_n]$ has a position $i$ such that $c$ equals $c_i$
    and, furthermore, the trie  $t_i$ associates a value with the key  $r$.
  - $ c \not\in C\!s \rightarrow
     \texttt{Trie}(v, C\!s, T\!s).\texttt{find}(cr) = \Omega
    $

    If $c$ does not occur in the array $C\!s$, then the trie $\texttt{Trie}(v, C\!s, T\!s)$ does not store a value for
    the key $cr$.

In [ ]:
Trie.prototype.find = function<T>(s: string): T | null {
  if (s === '') {
    return this.mValue;
  }
  const c = s[0];
  const r = s.slice(1);
  for (let i = 0; i < this.mChars.length; i++) {
    if (c === this.mChars[i]) {
      return this.mTries[i].find(r);
    }
  }
  return null;
};

The signature of the method $\mathtt{insert}$ is given as follows:
$$
\mathtt{insert}: \mathbb{T} \times \Sigma^* \times \mathtt{Value} \rightarrow \mathbb{T}.
$$

The result of evaluating  
$$
\mathtt{Trie}(v_1, [c_1, \cdots, c_n], [t_1, \cdots, t_n]).\mathtt{insert}(w, v_2)
$$
for a string $w \in \Sigma^*$ and a value $v_2 \in \mathtt{Value}$ is defined by induction on the
length of $w$:

- $\mathtt{Trie}(v_1,L,T).\mathtt{insert}(\varepsilon, v_2) = \mathtt{Trie}(v_2,L,T)$.  
  If a new value $v_2$ is associated with the empty string $\varepsilon$, then the old value
  $v_1$, which had been stored at the root before, is overwritten.

- $\mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_i,\cdots,c_n], [t_1,\cdots,t_i,\cdots,t_n]\bigr).\mathtt{insert}(c_ir,v_2) =
  \mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_i,\cdots,c_n], [t_1,\cdots,t_i.\mathtt{insert}(r,v_2),\cdots,t_n]\bigr)$.  
  In order to associate a value $v_2$ with the string $c_ir$ in the trie
  $$
  \mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_i,\cdots,c_n], [t_1,\cdots,t_i,\cdots,t_n]\bigr)
  $$
  we have to recursively associate the value $v_2$ with the string $r$ in the trie $t_i$.

- $c \notin \{c_1,\cdots,c_n\} \;\Rightarrow\;
  \mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_n], [t_1,\cdots,t_n]\bigr).\mathtt{insert}(cr,v_2) =
  \mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_n,c], [t_1,\cdots,t_n,\mathtt{Trie}(\Omega,[],[]).\mathtt{insert}(r,v_2)]\bigr)$.  
  If we want to associate a value $v_2$ with the key $cr$ in the trie
  $\mathtt{Trie}\bigl(v_1,[c_1,\cdots,c_n], [t_1,\cdots,t_n]\bigr)$, then, if the character $c$
  does not occur in the array $[c_1,\cdots,c_n]$, we first have to create a new empty trie.
  This trie has the form  
  $$
  \mathtt{Trie}(\Omega, [], []).
  $$
  Next, we associate the value $v_2$ with the key $r$ in this empty trie. Finally,
  we append the character $c$ to the end of the array $[c_1,\cdots,c_n]$ and append the trie  
  $$
  \mathtt{Trie}(\Omega, [], []).\mathtt{insert}(r,v_2)
  $$
  to the end of the array $[t_1,\cdots,t_n]$.


In [ ]:
Trie.prototype.insert = function<T>(s: string, v: T): void {
  if (s === '') {
    this.mValue = v;
    return;
  }
  const c = s[0];
  const r = s.slice(1);
  for (let i = 0; i < this.mChars.length; i++) {
    if (c === this.mChars[i]) {
      this.mTries[i].insert(r, v);
      return;
    }
  }
  const t = new Trie();
  t.insert(r, v);
  t.mParent = c; 
  this.mChars.push(c);
  this.mTries.push(t);
};

In order to implement deletion in tries, we need the auxiliary function `isEmpty`.  Its signature is given as
$$ \texttt{isEmpty}: \mathbb{T} \rightarrow \mathbb{B}. $$
For a trie $t$, we have $t.\texttt{isEmpty}() = \texttt{True}$ if and only if the trie $t$ does not
store any key.  The following formula specifies the function $\texttt{isEmpty}$:
$$ \texttt{Trie}(v, C, T).\texttt{isEmpty}() = \mathtt{True} \Leftrightarrow v = \Omega \wedge C = []. $$
Note that the condition $C = []$ implies $T = []$. 

In [ ]:
Trie.prototype.isEmpty = function(): boolean {
  return this.mValue === null && this.mChars.length === 0;
};

For a trie  $t \in \mathbb{T}$ and a string $w \in \Sigma^*$ the value $t.\texttt{delete}(w)$
is defined by induction on the length of  $w$.
  - $\texttt{Trie}(v,C,T).\texttt{delete}(\varepsilon) = \texttt{Trie}(\Omega,C,T)$

    The value that is associated with the empty  string $\varepsilon$ is stored at the root of the
    trie where is can be deleted without further ado.
  - $\begin{array}[t]{ll}
     t_i.\texttt{delete}(r).\texttt{isEmpty}()   & \rightarrow \\
     \texttt{Trie}(v, [c_1,\cdots,c_i,\cdots,c_n],[t_1,\cdots,t_i,\cdots,t_n]).\texttt{delete}(c_ir) 
     & = \\
     \qquad 
     \texttt{Trie}(v, [c_1,\cdots,c_{i-1},c_{i+1},\cdots,c_n],[t_1,\cdots,t_{i-1},t_{i+1},\cdots,t_n]).
     \end{array}
    $

    If  the key that is to be deleted starts with the character $c_i$ and if deletion of  the key
    $r$ in the $i$th  trie $t_i$ yields an empty
    trie, then both the $i^\mathrm{th}$ character $c_i$ and the $i^\mathrm{th}$ trie $t_i$ are deleted.
  - $\begin{array}[t]{ll}
     \neg t_i.\texttt{delete}(r).\texttt{isEmpty}()   & \rightarrow \\
     \texttt{Trie}(v, [c_1,\cdots,c_i,\cdots,c_n],[t_1,\cdots,t_i,\cdots,t_n]).\texttt{delete}(c_ir) 
     & = \\
     \qquad \texttt{Trie}(v, [c_1,\cdots,c_i,\cdots,c_n],[t_1,\cdots,t_i.\texttt{delete}(r),\cdots,t_n]).
     \end{array}
    $

    If  the key that is to be deleted starts with the character $c_i$ and if deletion of  the key
    $r$ in the $i^\mathrm{th}$  trie $t_i$ yields a non-empty trie, then the key $r$ has to be deleted recursively
    in the trie $t_i$.
  - $c \notin C \rightarrow \texttt{Trie}(v, C, T).\texttt{delete}(cr) =
       \texttt{Trie}(v, C, T)$. 
       
    If  the key that is to be deleted starts with the character $c$ and  $c$ does not occur in
    the array of characters $C$, then the trie does not contain the key $cr$ and therefore there
    is nothing  to do:  The trie is left unchanged.

In [ ]:
Trie.prototype.delete = function(s: string): void {
  if (s === '') {
    this.mValue = null;
    return;
  }
  const c = s[0];
  const r = s.slice(1);
  for (let i = 0; i < this.mChars.length; i++) {
    if (c === this.mChars[i]) {
      this.mTries[i].delete(r);
      if (this.mTries[i].isEmpty()) {
        this.mChars.splice(i, 1);
        this.mTries.splice(i, 1);
      }
      return;
    }
  }
};

# String Completion

In order to implement string completion we define the function `allKeys`, which
has the following signature:
$$\texttt{allKeys}: \mathbb{T} \times \Sigma^* \rightarrow \texttt{Set}(\Sigma^*)$$
Given a trie $t$ and a string $p$, the function $t.\texttt{allKeys}(p)$ computes the set of all strings that
are stored as keys in the trie $t$.  Furthermore, the string $p$ is added as a prefix to all these string.
Therefore we specify the semantics of the function $\texttt{allKeys}$ as follows:
$$ t.\texttt{allKeys}(p) = \{ p+w \mid w \in t \}. $$
Here, $p+w$ denotes the concatenation of the strings $p$ and $w$ and the expression $w \in t$ is true iff 
$t.\texttt{find}(w) \not= \Omega$.

Given a trie $t$, the value $t.\texttt{allKeys}(p)$ is computed by induction on $t$.  There are two cases:
- $\texttt{Trie}(\Omega, [c_1, \cdots, c_n], [t_1,\cdots,t_n]).\texttt{allKeys}(p) = 
   \bigcup\limits_{i=1}^n t_i.\texttt{allKeys}(p+c_i) 
  $,
- $v \not= \Omega \rightarrow 
   \texttt{Trie}(v, [c_1, \cdots, c_n], [t_1,\cdots,t_n]).\texttt{allKeys}(p) = 
   \{p\} \cup \bigcup\limits_{i=1}^n t_i.\texttt{allKeys}(p+c_i) 
  $.

In [ ]:
Trie.prototype.allKeys = function(prefix = ''): Set<string> {
  const Result = new Set<string>();
  if (this.mValue !== null) {
    Result.add(prefix);
  }
  for (let i = 0; i < this.mChars.length; i++) {
    const ci = this.mChars[i];
    const subKeys = this.mTries[i].allKeys(prefix + ci);
    for (const key of subKeys) {
      Result.add(key);
    }
  }
  return Result;
};

Next, we specify the auxiliary function `replacePrefix` that has the following signature:
$$ \texttt{replacePrefix}: \mathbb{T} \times \Sigma^* \times \Sigma^* \rightarrow \texttt{Set}(\Sigma^*) $$
Given a trie $t$, a string $s$, and a string $p$, it returns the set of all strings that are used as keys in $t$
and, furthermore, have the prefix $s$.  Additionally, it replaces the prefix $s$ with the string $p$.  Therefore
we have
$$ t.\texttt{replacePrefix}(s, p) = \bigl\{ p+r \mid s + r \in t \bigr\}. $$
The second argument $p$ is needed in order to make the inductive definition of the function
`replacePrefix` work out.  This definition is given next:
- $t.\texttt{replacePrefix}(\varepsilon, p) = t.\texttt{allKeys}(p)$,
- $\texttt{Trie}\bigl(v, [c_1, \cdots, c_n], [t_1,\cdots,t_n]\bigr).\texttt{replacePrefix}(c_ir, p) = 
   t_i.\texttt{replacePrefix}(r, p) 
  $
- $c \not\in C\!s \rightarrow \texttt{Trie}(v, C\!s, T\!s).\texttt{replacePrefix}(cr, p) = \{\}$.

In [ ]:
Trie.prototype.replacePrefix = function(s: string, p: string): Set<string> {
  if (s === '') {
    return this.allKeys(p);
  }
  const c = s[0];
  const r = s.slice(1);
  for (let i = 0; i < this.mChars.length; i++) {
    if (c === this.mChars[i]) {
      return this.mTries[i].replacePrefix(r, p);
    }
  }
  return new Set<string>();
};

Finally, we specify the function 
$$\texttt{findPrefix}:\mathbb{T} \times \Sigma^* \rightarrow \texttt{Set}(\Sigma^*)$$
so that given a trie $T$ and a prefix $p$, the expression $T.\texttt{findPrefix}(p)$ finds all
strings $s \in T$ that have the prefix $p$, i.e. that can be written in the form $s=p+r$:
$$ T.\texttt{findPrefix}(p) = \bigl\{ p+r \mid p+r \in T \bigr\}. $$
Having defined the auxiliary function `replacePrefix`, the function `findPrefix` can be
implemented as follows:
$$ T.\texttt{findPrefix}(s) = T.\texttt{replacePrefix}(s, s). $$

In [ ]:
Trie.prototype.findPrefix = function(s: string): Set<string> {
  return this.replacePrefix(s, s);
};

# Graphical Representation via GraphViz

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

In [ ]:
Trie.prototype.toDot = async function<T>(): Promise<string> {
  Trie.sTrieCount = 0;
  const TrieDict: Record<string, Trie<T>> = {};
  this.assignIDs(TrieDict);
  let dot = 'digraph G {\nnode [shape=record style=rounded];\n';
  for (const [id, t] of Object.entries(TrieDict)) {
    if (t.mValue !== null) {
      dot += `${id} [label="{${t.mParent}|${t.mValue}}"];\n`;
    } else {
      dot += `${id} [label="${t.mParent}"];\n`;
    }
  }
  for (const [id, t] of Object.entries(TrieDict)) {
    for (const x of t.mTries) {
      dot += `${id} -> ${x.mID};\n`;
    }
  }
  dot += '}\n';
  return dot;
};

In [ ]:
Trie.prototype.assignIDs = function<T>(TrieDict: Record<string, Trie<T>>): void {
  Trie.sTrieCount++;
  this.mID = String(Trie.sTrieCount);
  TrieDict[this.mID] = this;
  for (const t of this.mTries) {
    t.assignIDs(TrieDict);
  }
};

# Testing

In [ ]:
import { display } from "tslab";

async function renderTrie<T>(t: Trie<T>): Promise<void> {
  const dot = await t.toDot();           
  const gv = await Graphviz.load();       
  const svg = gv.layout(dot, "svg", "dot"); 
  display.html(svg);                       
}

In [ ]:
const t = new Trie<number>();
await renderTrie(t);

In [ ]:
t.insert("Hans", 1);
await renderTrie(t);

In [ ]:
t.insert("Hanna", 2);
await renderTrie(t);

In [ ]:
t.insert("Hilde", 3);
await renderTrie(t);

In [ ]:
t.insert("Hugo", 4);
await renderTrie(t);

In [ ]:
t.insert('Hildegard', 6);
await renderTrie(t);

In [ ]:
t.findPrefix('Hi');

In [ ]:
t.insert("Harro", 5);
await renderTrie(t);

In [ ]:
t.insert("Heike", 6);
await renderTrie(t);

In [ ]:
t.allKeys('abc_');

In [ ]:
t.replacePrefix('H', 'X');

In [ ]:
t.insert("Heiko", 7);
await renderTrie(t);

In [ ]:
t.find('Hilde');

In [ ]:
t.insert("Harald", 8);
t.insert("Hasso", 9);
t.insert("Haley", 1);
t.insert("Hanny", 1);
t.insert("Happy", 1);
t.insert("Harlene", 1);
t.insert("Harley", 1);
t.insert("Harmony", 1);
t.insert("Harriet", 1);
t.insert("Harrietta", 1);
t.insert("Hazel", 1);
t.insert("Heather", 1);
t.insert("Hedwig", 1);
t.insert("Hedy", 1);
t.insert("Heidi", 1);
t.insert("Helaina", 1);
t.insert("Helaine", 1);
t.insert("Helen", 1);
t.insert("Helena", 1);
t.insert("Helene", 1);
t.insert("Helga", 1);
t.insert("Hendrika", 1);
t.insert("Henrietta", 1);
t.insert("Henriette", 1);
t.insert("Hermione", 1);
t.insert("Herta", 1);
t.insert("Hilary", 1);
t.insert("Hilda", 1);
await renderTrie(t);

In [ ]:
t.findPrefix('Hea');

In [ ]:
t.findPrefix('Hel');

In [ ]:
t.delete('Hanny');
await renderTrie(t);

In [ ]:
t.allKeys();

In [ ]:
t.delete("Hans");
t.delete("Hanna");
t.delete("Hilde");
t.delete("Hugo");
t.delete("Harro");
t.delete("Heike");
t.delete("Heiko");
t.delete("Harald");
t.delete("Hasso");
t.delete("Haley");
t.delete("Hanny");
t.delete("Happy");
t.delete("Harlene");
t.delete("Harley");
t.delete("Harmony");
t.delete("Harriet");
t.delete("Harrietta");
t.delete("Hazel");
t.delete("Heather");
t.delete("Hedwig");
t.delete("Hedy");
t.delete("Heidi");
t.delete("Helaina");
t.delete("Helaine");
t.delete("Helen");
t.delete("Helena");
t.delete("Helene");
t.delete("Helga");
t.delete("Hendrika");
t.delete("Henrietta");
t.delete("Henriette");
t.delete("Hermione");
t.delete("Herta");
t.delete("Hilary");
t.delete("Hilda");
t.delete("Hildegard");
await renderTrie(t);

In order to make sure that everything works, let us compute the prime numbers less than $100$.  
Note that we have to convert numbers to strings in order to be able to use tries.

In [ ]:
const S = new Trie<number>();
for (let i = 2; i <= 100; i++) {
  S.insert(i.toString(), i);
}
for (let i = 2; i <= 50; i++) {
  for (let j = i; j <= Math.floor(100 / i); j++) {
    S.delete((i * j).toString());
  }
}
await renderTrie(S);

Let us compute all key stored in this trie, as these are the prime numbers.

In [ ]:
const keys = Array.from(S.allKeys()).map(p => parseInt(p, 10)).sort((a, b) => a - b);
console.log(keys);